# Sumativa 3 — MCDI501: Estadística Computacional para la Toma de Decisiones
## Cierre y comunicación del proyecto — Fase 4

**Integrantes:** «Daniel Hormazábal», «Cristian Pastén»
**Docente:** «Jean Paul Maidana»
**Fecha:** «14/07/2026»

Semilla utilizada para reproducibilidad: `123456`

Este notebook integra el trabajo de S1 (análisis exploratorio e inferencial) y S2 (validación mediante remuestreo y simulación) para construir modelos de regresión lineal (imputación) y regresión logística (clasificación), siguiendo la progresión S1 → S2 → S3 solicitada en el enunciado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_SEED = 123456
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
plt.rcParams.update({'font.size': 10})

## Parte 1 — Manejo inteligente de datos faltantes

### 1.1 Referencia al análisis de datos faltantes de la Sumativa 1

En S1 se verificó la ausencia de valores nulos estándar (`NaN`) en las 4.424 filas y 37 variables del dataset ("Predict Students' Dropout and Academic Success", Realinho et al., 2021). Sin embargo, ese chequeo estándar no detecta **faltantes disfrazados como códigos categóricos válidos** — un patrón documentado en el codebook oficial del dataset (UCI/Zenodo DOI: 10.5281/zenodo.5777340).

Se revisó el codebook completo y se identificaron dos categorías explícitas de faltante disfrazado:
- `Mother's occupation` / `Father's occupation`: código **99 = "(blank)"**
- `Mother's qualification` / `Father's qualification`: código **34 = "Unknown"**

Ninguna otra variable del dataset (37 en total) presenta una categoría de faltante documentada en el codebook oficial.

In [ ]:
df = pd.read_csv('../data/predict_students_dropout_and_academic_success.csv', sep=';')
df.columns = df.columns.str.strip()
print("Dimensiones:", df.shape)

# Chequeo estándar (ya confirmado en S1): sin NaN
print("\nNulos estándar (NaN):")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Chequeo de strings vacíos en columnas tipo objeto
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    vacios = df[col].astype(str).str.strip().eq('').sum()
    if vacios > 0:
        print(f"{col}: {vacios} strings vacíos")

# Faltantes disfrazados documentados en el codebook oficial
print("\nFaltantes disfrazados (según codebook UCI):")
print("Mother's occupation == 99 (blank):", (df["Mother's occupation"] == 99).sum())
print("Father's occupation == 99 (blank):", (df["Father's occupation"] == 99).sum())
print("Mother's qualification == 34 (Unknown):", (df["Mother's qualification"] == 34).sum())
print("Father's qualification == 34 (Unknown):", (df["Father's qualification"] == 34).sum())

**Resumen de faltantes reales identificados:**

| Variable | Código sentinela | Casos | % del total |
|---|---|---|---|
| Mother's qualification | 34 = "Unknown" | 130 | 2,94% |
| Father's qualification | 34 = "Unknown" | 112 | 2,53% |
| Mother's occupation | 99 = "(blank)" | 17 | 0,38% |
| Father's occupation | 99 = "(blank)" | 19 | 0,43% |

### 1.2 Patrón de faltantes: MCAR, MAR o MNAR

Se recodificaron los códigos sentinela como `NaN` y se analizó el solapamiento entre faltantes de `Mother's qualification` y sus posibles predictoras.

In [ ]:
df_num = df.copy()
df_num["Mother's qualification"] = df_num["Mother's qualification"].replace(34, np.nan)
df_num["Father's qualification"] = df_num["Father's qualification"].replace(34, np.nan)
df_num["Mother's occupation"] = df_num["Mother's occupation"].replace(99, np.nan)
df_num["Father's occupation"] = df_num["Father's occupation"].replace(99, np.nan)

# Solapamiento de faltantes: evidencia de MAR
mask_target_missing = df_num["Mother's qualification"].isna()
candidatas_check = ["Father's qualification", "Age at enrollment", "Mother's occupation"]
print("De 130 filas con Mother's qualification faltante:")
for p in candidatas_check:
    solapa = df_num[mask_target_missing][p].isna().sum()
    print(f"  {p}: {solapa} también faltante")

**Interpretación:** el 77% (100/130) de los casos con `Mother's qualification` faltante también tienen `Father's qualification` faltante — una co-ocurrencia sistemática que descarta MCAR (falta completamente al azar). El patrón es consistente con **MAR** (Missing At Random): la ausencia está condicionada a un factor observable (probablemente familias que no reportan antecedentes educativos de ningún progenitor), no al valor faltante en sí mismo. Esto justifica metodológicamente el uso de imputación por regresión, que explota la información de otras variables observadas.

### 1.3 Selección de predictoras mediante matriz de correlaciones

Siguiendo el criterio de S1 (matriz de correlaciones) extendido a estas nuevas variables, se calculó la correlación de `Mother's qualification` con las variables numéricas candidatas.

In [ ]:
candidatas = ["Age at enrollment", "Admission grade", "Previous qualification (grade)",
              "Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)",
              "Father's qualification", "Mother's occupation", "Father's occupation",
              "Unemployment rate", "Inflation rate", "GDP"]

corr_madre = df_num[candidatas + ["Mother's qualification"]].corr(numeric_only=True)["Mother's qualification"].drop("Mother's qualification")
print(corr_madre.sort_values(key=abs, ascending=False))

`Father's qualification` tiene la correlación más fuerte (r=0,529), seguida de `Age at enrollment` (r=0,286). Sin embargo, `Father's qualification` solapa con el 77% de los faltantes de `Mother's qualification` (ver 1.2), lo que reduciría drásticamente la cobertura de imputación. Se prioriza por tanto un set de predictoras sin solapamiento relevante: **Age at enrollment, Admission grade, Mother's occupation** — priorizando cobertura sobre poder explicativo puro, documentando el trade-off.

### 1.4 Imputación mediante regresión lineal múltiple — Mother's qualification

In [ ]:
predictoras_v2 = ["Age at enrollment", "Admission grade", "Mother's occupation"]
target = "Mother's qualification"

train_v2 = df_num[[target] + predictoras_v2].dropna()
print("Filas de entrenamiento:", len(train_v2))

X2 = sm.add_constant(train_v2[predictoras_v2])
y2 = train_v2[target]

modelo_madre_qual = sm.OLS(y2, X2).fit()
print(modelo_madre_qual.summary())

In [ ]:
# VIF (incluyendo la constante en la matriz, requisito del cálculo correcto)
vif_data = pd.DataFrame()
vif_data["variable"] = X2.columns
vif_data["VIF"] = [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]
print(vif_data)
# Nota: el VIF de 'const' se ignora siempre; es un artefacto sin interpretación causal.

In [ ]:
# Diagnóstico de supuestos
residuos = modelo_madre_qual.resid
fitted = modelo_madre_qual.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(fitted, residuos, alpha=0.3)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuos vs. ajustados')

stats.probplot(residuos, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q plot (normalidad)')

axes[2].hist(residuos, bins=30)
axes[2].set_title('Histograma de residuos')
plt.tight_layout()
plt.savefig('diagnostico_imputacion_mother_qualification.png', dpi=100)
plt.show()

print("Shapiro-Wilk p-valor:", stats.shapiro(residuos.sample(min(5000, len(residuos)), random_state=RANDOM_SEED))[1])

**Limitación metodológica reconocida:** los residuos muestran bandas diagonales paralelas y el histograma es multimodal (no campana) — consecuencia esperada de usar regresión OLS sobre una variable que es, en su naturaleza, un **código categórico nominal** (niveles educativos sin distancia real entre categorías) y no una escala continua. Shapiro-Wilk confirma no-normalidad (p≈0), pero se documenta como limitación reconocida y no como error del proceso: el enunciado exige explícitamente este método para variables numéricas, y el dataset codifica estas variables como numéricas.

### 1.5 Aplicación de la imputación: redondeo al código válido más cercano

Dado que `Mother's qualification` es un código categórico discreto, las predicciones continuas del modelo se redondean al **código válido más cercano** dentro del catálogo real de valores observados (no al entero más cercano sin más, que podría caer en un código inexistente).

In [ ]:
def redondear_a_codigo_valido(valor_continuo, catalogo):
    """Encuentra el código válido más cercano al valor continuo predicho."""
    catalogo_arr = np.array(catalogo)
    idx_mas_cercano = np.argmin(np.abs(catalogo_arr - valor_continuo))
    return catalogo_arr[idx_mas_cercano]

mask_imputar = df_num["Mother's qualification"].isna() & df_num[predictoras_v2].notna().all(axis=1)
print(f"Filas a imputar por regresión: {mask_imputar.sum()}")

X_imputar = sm.add_constant(df_num.loc[mask_imputar, predictoras_v2], has_constant='add')
predicciones = modelo_madre_qual.predict(X_imputar)

codigos_validos = sorted(df["Mother's qualification"].dropna().unique())
codigos_validos = [c for c in codigos_validos if c != 34]
predicciones_redondeadas = predicciones.apply(lambda v: redondear_a_codigo_valido(v, codigos_validos))

df_num["Mother's qualification_imputada"] = df_num["Mother's qualification"].copy()
df_num.loc[mask_imputar, "Mother's qualification_imputada"] = predicciones_redondeadas

print("\nDistribución de códigos imputados vía regresión:")
print(predicciones_redondeadas.value_counts().sort_index())

In [ ]:
# Casos residuales (target y alguna predictora faltante simultáneamente): imputación por moda global
faltantes_residuales = df_num["Mother's qualification_imputada"].isna()
moda_global_madre = df["Mother's qualification"][df["Mother's qualification"] != 34].mode()[0]
print(f"Moda global (excluyendo 'Unknown'): código {moda_global_madre}")
print(f"Casos residuales imputados con moda: {faltantes_residuales.sum()}")

df_num.loc[faltantes_residuales, "Mother's qualification_imputada"] = moda_global_madre
print("Valores sin imputar restantes:", df_num["Mother's qualification_imputada"].isna().sum())

**Hallazgo relevante para el análisis comparativo (Parte 3):** los valores imputados por regresión se concentran en códigos 14-39, y **ninguno cae en las categorías más frecuentes de la población real** (código 1 = "Secundaria", 24,9%; código 3 = "Educación superior", 10,2%). Esto ocurre porque, con edad mínima de 17 años, el valor mínimo que el modelo puede predecir estructuralmente ronda ~20 — la imputación por regresión sistemáticamente **distorsiona la distribución hacia categorías intermedias**, subrepresentando la categoría más común de la población. Este es un hallazgo cuantificable, no un defecto oculto.

### 1.6 Imputación de Father's qualification, Mother's occupation y Father's occupation

Se aplica la misma metodología de forma encadenada: las variables ya imputadas se usan como predictoras de los pasos siguientes (manteniendo el vínculo temático educación↔ocupación sin perder cobertura por solapamiento de faltantes).

In [ ]:
def imputar_por_regresion(df_num, df_original, target, predictoras, codigo_excluir):
    train = df_num[[target] + predictoras].dropna()
    X = sm.add_constant(train[predictoras])
    y = train[target]
    modelo = sm.OLS(y, X).fit()

    vif = pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    })

    mask_imputar = df_num[target].isna() & df_num[predictoras].notna().all(axis=1)
    X_pred = sm.add_constant(df_num.loc[mask_imputar, predictoras], has_constant='add')
    predicciones = modelo.predict(X_pred)

    codigos_validos = sorted(df_original[target].dropna().unique())
    codigos_validos = [c for c in codigos_validos if c != codigo_excluir]
    predicciones_redondeadas = predicciones.apply(lambda v: redondear_a_codigo_valido(v, codigos_validos))

    col_imputada = f"{target}_imputada"
    df_num[col_imputada] = df_num[target].copy()
    df_num.loc[mask_imputar, col_imputada] = predicciones_redondeadas

    faltantes_residuales = df_num[col_imputada].isna()
    moda = df_original[target][df_original[target] != codigo_excluir].mode()[0]
    df_num.loc[faltantes_residuales, col_imputada] = moda

    print(f"=== {target} ===")
    print(f"R²={modelo.rsquared:.3f} | Cobertura regresión: {mask_imputar.sum()} | Moda residual: {faltantes_residuales.sum()}")
    print(vif)
    print(modelo.params)
    print()

    return df_num, modelo, vif

df_num, modelo_padre_qual, vif_padre_qual = imputar_por_regresion(
    df_num, df, "Father's qualification",
    ["Age at enrollment", "Admission grade", "Father's occupation"],
    codigo_excluir=34
)

df_num, modelo_madre_ocup, vif_madre_ocup = imputar_por_regresion(
    df_num, df, "Mother's occupation",
    ["Age at enrollment", "Admission grade", "Mother's qualification_imputada"],
    codigo_excluir=99
)

df_num, modelo_padre_ocup, vif_padre_ocup = imputar_por_regresion(
    df_num, df, "Father's occupation",
    ["Age at enrollment", "Admission grade", "Father's qualification_imputada"],
    codigo_excluir=99
)

### 1.7 Resumen de la imputación

| Variable | Faltantes totales | Vía regresión | Vía moda | R² del modelo |
|---|---|---|---|---|
| Mother's qualification | 130 | 116 | 14 | 0,086 |
| Father's qualification | 112 | 97 | 15 | 0,037 |
| Mother's occupation | 17 | 17 | 0 | 0,007 |
| Father's occupation | 19 | 19 | 0 | 0,004 |

VIF ≈1 en todas las predictoras de los cuatro modelos — sin colinealidad. Los R² de las variables de ocupación son muy bajos (≈0), un hallazgo honesto a documentar: los códigos de ocupación (clasificación tipo ISCO, no ordinal) no son bien explicados por edad, nota de admisión ni nivel educativo del progenitor. El impacto práctico es acotado dado el bajo número de casos afectados (17 y 19).

**Pendiente:** Parte 2 (regresión logística), Parte 3 (comparación de estrategias de imputación), Parte 4 (informe integrado), Parte 5 (presentación).